# starry: TIC 427395536, sector 6

Ajuste de binaria eclipsante con starry + PyMC3. Los priors salen de
`4_pipeline_completo.ipynb` (carpeta `resultados_pipeline/`) y de `priors_v2.csv`
(profundidades medidas en el fondo del eclipse, ver `1b_profundidades_eclipse_v2.ipynb`). Generado automáticamente; los valores de los priors están
escritos en la sección **Priors** y se pueden editar a mano.

- Archivo: `tic 427395536_S_6_LC.txt` (911 puntos, cadencia ~30.0 min)
- Primaria: rama **MIST**, edad 0.166 Myr

**Avisos:**

- ⚠️ P_orb de este sector no coincide con otros sectores: S32: 0.73387 d, S98: 2.33697 d (aquí 4.67394 d). Revisar si es P, 2P o P/2.
- ⚠️ La profundidad primaria cambia entre sectores más de lo que explican sus errores (chi2_red = 103: S6: 0.044, S32: 0.036, S98: 0.060). Suele ser dilución/contaminación distinta por sector o suavizado por cadencia.

Todas las salidas se guardan en `starry_por_tic/TIC_427395536/S6_ecc/`. El trace se guarda en `.nc`, así que
para rehacer figuras sin volver a muestrear basta con `CARGAR_TRACE_EXISTENTE = True`.

In [ ]:
# =========================
# Básicas
# =========================
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import animation
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from IPython.display import HTML, display

# =========================
# Light curves
# =========================
from wotan import flatten
from astropy.timeseries import LombScargle

# =========================
# Modelado
# =========================
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import starry
import theano
import theano.tensor as tt
import arviz as az

# =========================
# Utilidades
# =========================
from scipy.stats import median_abs_deviation
import corner

In [ ]:
# =========================
# Configuración
# =========================
TIC, SECTOR = 427395536, 6
Label = f"TIC_{TIC}_S{SECTOR}"

FILE_LC = Path("/home/luciam/Documents/starry/Resultados_Orion/Light_Curve_Files/tic 427395536_S_6_LC.txt")
OUT_DIR = Path("/home/luciam/Documents/starry/Resultados_Orion/starry_por_tic/TIC_427395536/S6_ecc")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "Figures"
FIG_DIR.mkdir(exist_ok=True)

# Grado de los mapas. Con 0 las estrellas son discos uniformes con limb darkening: es el mismo
# modelo que ydeg=5 con todos los coeficientes y en cero (como en el ejemplo, donde nunca se
# ajustan), pero ~70 veces más rápido. Usar > 0 solo si se van a muestrear los coeficientes y (manchas).
YDEG = 0

WINDOW_WOTAN = 0.55     # días, ventana de wotan (igual que el ejemplo)
DETRENDING_CON_MASCARA = True   # tendencia calculada sin los eclipses (False = como el ejemplo)
BIN_MIN = None          # None = sin binear; p. ej. 10 -> bins de 10 min (acelera mucho S87/S98)

# muestreo
TUNE, DRAWS = 1500, 1500
CHAINS, CORES = 4, 5
TARGET_ACCEPT = 0.9
CARGAR_TRACE_EXISTENTE = True   # si ya existe el .nc de este TIC/sector, se carga en vez de muestrear
N_CURVAS = 300                  # muestras del posterior para las bandas 16-84 % del modelo

# ¿B_r libre (como el ejemplo) o B_r = A_r * k con k ~ Normal(sqrt(Depth_primary), sigma)?
# La segunda opción usa directamente Depth_primary = (R_B/R_A)^2 y no cuenta dos veces el error de R_A.
RADIO_B_COMO_RAZON = False

# Tiempo de integración de cada punto. None = flujo instantáneo (como el ejemplo). Con cadencia
# de 30 min conviene TEXP = 0.02083 (días); starry integra el modelo durante la exposición
# (más lento, ~oversample veces).
TEXP = None

HACER_ANIMACION = True   # requiere ffmpeg
TRACE_PATH = OUT_DIR / f"{Label}_trace.nc"

np.random.seed(12)
starry.config.lazy = False
starry.config.quiet = True

## 1) Datos y detrending

In [ ]:
data = pd.read_csv(FILE_LC, sep=r"\s+").rename(columns={"#Time": "Time"})
data = data.dropna(subset=["Time", "Flux_CBV_SS"]).sort_values("Time").drop_duplicates("Time")

time = data["Time"].to_numpy(float)
flux = data["Flux_CBV_SS"].to_numpy(float)

# Detrending con los eclipses enmascarados (1b_profundidades_eclipse_v2, sección 2.1): la tendencia
# biweight se calcula solo fuera de los eclipses y se interpola sobre ellos, para que no "se coma"
# parte de la profundidad ni deje picos falsos junto a los eclipses.
P_MASK, T0_MASK, DUR_MASK, FASE2_MASK = 4.67393697, 1469.794179, 0.179, 0.5017
W_mask = min(DUR_MASK / P_MASK, 0.2)
ph_mask = ((time - T0_MASK) / P_MASK) % 1
dist = lambda c: np.abs((ph_mask - c + 0.5) % 1 - 0.5)
fuera = (dist(0.0) > W_mask) & (dist(FASE2_MASK) > W_mask)
if DETRENDING_CON_MASCARA and fuera.sum() > 100:
    _, tr = flatten(time[fuera], flux[fuera], window_length=WINDOW_WOTAN, break_tolerance=1.5,
                    method="biweight", return_trend=True)
    ok_tr = np.isfinite(tr)
    trend_flux = np.interp(time, time[fuera][ok_tr], tr[ok_tr])
    flat_flux = flux / trend_flux
else:
    flat_flux, trend_flux = flatten(time, flux, window_length=WINDOW_WOTAN, break_tolerance=1.5,
                                    method="biweight", return_trend=True)
ok = np.isfinite(flat_flux)
time, flux, flat_flux, trend_flux = time[ok], flux[ok], flat_flux[ok], trend_flux[ok]

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].scatter(time, flux, s=5, color="k", label="Original")
ax[0].plot(time, trend_flux, color="r", lw=2, label="Trend")
ax[0].set_ylabel("Flux"); ax[0].legend(); ax[0].set_title("Wotan detrending")
ax[1].scatter(time, flat_flux, s=4.5, color="b")
ax[1].set_xlabel("Time [days]"); ax[1].set_ylabel("Normalised flux")
plt.tight_layout(); plt.show()

pd.DataFrame({"time": time, "flat_flux": flat_flux, "trend_flux": trend_flux}).to_csv(
    OUT_DIR / f"{Label}_trending.csv", index=False)

In [ ]:
# =========================
# Datos para el ajuste (opcionalmente bineados)
# =========================
t_real, flux_real = time, flat_flux
if BIN_MIN:
    idx = np.floor((t_real - t_real[0]) / (BIN_MIN / 1440)).astype(int)
    b = pd.DataFrame({"t": t_real, "f": flux_real, "i": idx}).groupby("i").mean()
    t_real, flux_real = b["t"].to_numpy(), b["f"].to_numpy()

sigma_sd = 1.4826 * median_abs_deviation(flux_real)
print(f"{len(t_real)} puntos para el ajuste (de {len(time)}), sigma_sd = {sigma_sd:.2e}")

fig, ax = plt.subplots(1, figsize=(12, 5))
ax.plot(t_real, flux_real, ".k", alpha=0.6, ms=4)
ax.set_xlabel("time [days]", fontsize=16); ax.set_ylabel("normalized flux", fontsize=16);

## 2) Priors

- **Primaria**: rama MIST del ajuste massage: R_A = Stefan-Boltzmann(T_median_1, lgL_median_1), M_A = mass_median_1 (sd = mass_std_1, mínimo 10 %).
- **R_B** = R_A·sqrt(Depth_primary), Depth_primary (v2) = 0.04418 ± 0.00203 (v1 era 0.01662; sesgo sistemático máximo del estimador v2: 5%). sd_r_B propaga el error de R_A y el de la profundidad.
- **M_B**: BHAC15 a la edad de Somers17 (M2 que reproduce R2 en la isocrona).
- **amp**: amp_B = Depth_secondary v2 (eclipse secundario detectado a 20 sigma; en un eclipse secundario total la profundidad es la fracción de luz de B), sd = max(error, 10 %); estimación de cuerpo negro en banda TESS: 0.0120. amp_A = 1 − amp_B.
- **P_orb, t0**: priors de la curva de luz de este sector (sd_t0 = duración del eclipse).
- Teff_A = 3485 K, Teff_B = 2798 K.
- **ecc, w**: prior no informativo (uniforme en e y en w). Secundario en fase ~0.5 (|e cos w| < 0.03) o no detectado: se empieza casi circular (e0 = 0.001), w poco restringido.
- inc, u: mismos valores genéricos del notebook de ejemplo.

In [ ]:
# =========================
# Priors (generados desde resultados_pipeline/; editar aquí si hace falta)
# =========================
# Primaria
inc_A, sd_inc_A = 88.0, 1
amp_A, sd_amp_A = 0.92107, 0.0078933
r_A, sd_r_A = 2.6935, 0.32259
m_A, sd_m_A = 0.313, 0.062
u_A = [0.3, 0.2]

# Secundaria
inc_B, sd_inc_B = 88.0, 1
amp_B, sd_amp_B = 0.078933, 0.0078933
r_B, sd_r_B = 0.56615, 0.069039        # R_A * sqrt(Depth_primary), Depth_primary (v2) = 0.044179 ± 0.0020287
k_B, sd_k_B = 0.21019, 0.004826            # R_B/R_A = sqrt(Depth_primary) (solo si RADIO_B_COMO_RAZON)
m_B, sd_m_B = 0.044133, 0.030901
porb_B, sd_porb_B = 4.67393697, 0.01
t0_B, sd_t0_B = 1469.794179, 0.179          # sd = duración del eclipse (BLS)
# Excentricidad: SIN prior informativo. Se muestrea (sqrt(e) cos w, sqrt(e) sin w) uniforme en el
# disco unitario (sección 4). Estos dos números son solo el punto de partida (testval).
ecc_B, w_B = 0.001, 90
u_B = [0.3, 0.2]

prot_s = 0

## 3) Modelo inicial

In [ ]:
A_y = np.zeros((YDEG + 1) ** 2 - 1)   # coeficientes y (sin y00): mapa uniforme
B_y = np.zeros((YDEG + 1) ** 2 - 1)

A = dict(ydeg=YDEG, udeg=2, inc=inc_A, amp=amp_A, r=r_A, m=m_A, prot=prot_s, u=u_A, y=A_y)
B = dict(ydeg=YDEG, udeg=2, inc=inc_B, amp=amp_B, r=r_B, m=m_B, prot=prot_s, porb=porb_B, t0=t0_B,
         ecc=ecc_B, w=w_B, u=u_B, y=B_y)

pri = starry.Primary(starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A["inc"], amp=A["amp"]),
                     r=A["r"], m=A["m"], prot=A["prot"])
pri.map[1:] = A["u"]
if YDEG > 0:
    pri.map[1:, :] = A["y"]

sec = starry.Secondary(starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B["inc"], amp=B["amp"]),
                       r=B["r"], m=B["m"], porb=B["porb"], prot=B["prot"], t0=B["t0"],
                       inc=B["inc"], ecc=B["ecc"], w=B["w"])
sec.map[1:] = B["u"]
if YDEG > 0:
    sec.map[1:, :] = B["y"]

sys = starry.System(pri, sec, texp=TEXP)
flux_true = sys.flux(t_real)

fig, ax = plt.subplots(1, figsize=(16, 7))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=6, label="datos")
ax.plot(t_real, flux_true, lw=2, label="modelo con los priors")
ax.set_xlabel("time [days]", fontsize=16); ax.set_ylabel("normalized flux", fontsize=16); ax.legend();

## 4) Modelo PyMC3

Diferencias con el notebook de ejemplo:

- Los mapas usan `ydeg = YDEG` (0 por defecto) en lugar de 5. Como los coeficientes y nunca se
  ajustan (están en cero), el modelo es el mismo (se verificó que el flujo coincide hasta
  ~10⁻¹⁵), pero cada evaluación de logp + gradiente es ~70 veces más rápida.
- Los coeficientes de limb darkening `u_A`, `u_B` (`QuadLimbDark`) ahora **sí** entran en el
  mapa.
- **Excentricidad.** No se muestrean $e$ y $\omega$ directamente, sino

  $$h=\sqrt e\cos\omega,\qquad k=\sqrt e\sin\omega,\qquad (h,k)\sim\mathcal U(\text{disco unitario}),$$

  y $e=h^2+k^2$, $\omega=\operatorname{atan2}(k,h)$ se guardan como `Deterministic`
  (Eastman, Gaudi & Agol 2013).

  *Prior implícito.* En polares $(r,\omega)$ con $r=\sqrt e$, el elemento de área es
  $dh\,dk=r\,dr\,d\omega$, y como $dr=de/(2\sqrt e)$ queda $dh\,dk=\tfrac12\,de\,d\omega$. Así que un
  prior uniforme en el disco equivale a $e\sim\mathcal U(0,1)$ y $\omega\sim\mathcal U(-\pi,\pi)$,
  independientes: **no favorece ninguna excentricidad**.

  *Por qué ayuda al muestreo.*
  1. En $(e,\omega)$, el punto $e=0$ es un borde donde $\omega$ no está definido. Las cadenas se
     amontonan contra él y aparecen divergencias (en las corridas anteriores: 99–725). En
     $(h,k)$, $e=0$ es el centro del disco, un punto interior sin nada especial.
  2. Los datos fijan muy bien $e\cos\omega$ (por el momento del secundario) y mal $e\sin\omega$ (por
     las duraciones). En $(e,\omega)$ eso es una curva angosta con correlación ≈ 1 entre $e$ y
     $\omega$. En $(h,k)$ es una franja casi recta, que la matriz de masa densa de NUTS sí puede
     adaptar.

  *Sin doble conteo.* El $e\cos\omega$ medido en `1b` sale de **esta misma curva de luz**, así que
  ya está en la verosimilitud. Usarlo también como prior (como antes: $e\sim\mathcal N(|e\cos\omega|,\,\cdot)$,
  $\omega\sim\mathcal N(0°,60°)$) lo contaba dos veces y encima jalaba $\omega$ hacia 0°. Ahora solo
  se usa como **punto de partida** del optimizador y del muestreo. En el ejemplo se muestreaban pero el
mapa usaba los valores fijos `[0.3, 0.2]`, así que su posterior era solo el prior.

In [ ]:
starry.config.lazy = True
starry.config.quiet = True

with pm.Model() as model:
    PositiveNormal = pm.Bound(pm.Normal, lower=0.0)

    # Primaria
    A_inc = pm.Normal("A_inc", mu=inc_A, sd=sd_inc_A, testval=inc_A)
    A_amp = pm.Normal("A_amp", mu=amp_A, sd=sd_amp_A, testval=amp_A)
    A_r = PositiveNormal("A_r", mu=r_A, sd=sd_r_A, testval=r_A)
    A_m = PositiveNormal("A_m", mu=m_A, sd=sd_m_A, testval=m_A)
    A_u = xo.distributions.QuadLimbDark("u_A", testval=np.array(A["u"]))
    pri = starry.Primary(starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A_inc, amp=A_amp),
                         r=A_r, m=A_m, prot=prot_s)
    pri.map[1] = A_u[0]
    pri.map[2] = A_u[1]

    # Secundaria
    B_inc = pm.Normal("B_inc", mu=inc_B, sd=sd_inc_B, testval=inc_B)
    B_amp = pm.Normal("B_amp", mu=amp_B, sd=sd_amp_B, testval=amp_B)
    if RADIO_B_COMO_RAZON:
        B_k = PositiveNormal("B_k", mu=k_B, sd=sd_k_B, testval=k_B)
        B_r = pm.Deterministic("B_r", A_r * B_k)
    else:
        B_r = PositiveNormal("B_r", mu=r_B, sd=sd_r_B, testval=r_B)
    B_m = PositiveNormal("B_m", mu=m_B, sd=sd_m_B, testval=m_B)
    B_porb = PositiveNormal("B_porb", mu=porb_B, sd=sd_porb_B, testval=porb_B)
    B_t0 = pm.Normal("B_t0", mu=t0_B, sd=sd_t0_B, testval=t0_B)
    B_u = xo.distributions.QuadLimbDark("u_B", testval=np.array(B["u"]))
    # (h, k) = (sqrt(e) cos w, sqrt(e) sin w), uniforme en el disco unitario -> e ~ U(0,1), w ~ U(-180,180)
    hk0 = np.sqrt(ecc_B) * np.array([np.cos(np.deg2rad(w_B)), np.sin(np.deg2rad(w_B))])
    B_hk = pmx.UnitDisk("B_hk", testval=hk0)
    B_ecc = pm.Deterministic("B_ecc", tt.sum(B_hk ** 2))
    B_w = pm.Deterministic("B_w", tt.arctan2(B_hk[1], B_hk[0]) * 180.0 / np.pi)
    sec = starry.Secondary(starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B_inc, amp=B_amp),
                           r=B_r, m=B_m, porb=B_porb, prot=prot_s, t0=B_t0, inc=B_inc, ecc=B_ecc, w=B_w)
    sec.map[1] = B_u[0]
    sec.map[2] = B_u[1]

    sys = starry.System(pri, sec, texp=TEXP)
    flux_model = sys.flux(t=t_real)
    flux_model_guess = pmx.eval_in_model(flux_model)

    pm.Normal("obs", mu=flux_model, sd=sigma_sd, observed=flux_real)

var_names_A = ["A_m", "A_r", "A_inc", "u_A", "A_amp"]
var_names_B = ["B_m", "B_r", "B_porb", "B_inc", "B_t0", "B_hk", "B_ecc", "B_w", "u_B", "B_amp"]

In [ ]:
with model:
    map_soln = pmx.optimize(start=model.test_point)

plt.figure(figsize=(12, 5))
plt.plot(t_real, flux_real, "k.", alpha=0.3, ms=2, label="data")
plt.plot(t_real, flux_model_guess, "C1--", lw=1, alpha=0.5, label="Initial")
plt.plot(t_real, pmx.eval_in_model(flux_model, map_soln, model=model), "C1-", label="MAP", lw=1)
plt.legend(fontsize=10, numpoints=5)
plt.xlabel("time [days]", fontsize=16); plt.ylabel("relative flux", fontsize=16);
{k: np.round(v, 5) for k, v in map_soln.items() if not k.endswith("__")}

## 5) Muestreo

`flux_model` no se guarda como Deterministic (en S98 serían ~0.8 GB). Las curvas del
modelo se recalculan después a partir de `N_CURVAS` muestras del posterior.

In [ ]:
if CARGAR_TRACE_EXISTENTE and TRACE_PATH.exists():
    trace = az.from_netcdf(TRACE_PATH)
    print(f"Trace cargado de {TRACE_PATH}")
else:
    with model:
        mtrace = pmx.sample(tune=TUNE, draws=DRAWS, start=map_soln, cores=CORES, chains=CHAINS,
                            target_accept=TARGET_ACCEPT, return_inferencedata=False)
        trace = az.from_pymc3(mtrace)
    trace.to_netcdf(TRACE_PATH)
    print(f"Trace guardado en {TRACE_PATH}")

summary = az.summary(trace, var_names=var_names_A + var_names_B)
summary.to_csv(OUT_DIR / f"{Label}_Summary.csv")
display(summary)

## 6) Posteriors

In [ ]:
post = trace.posterior
inf_pos = {n: post[n].values.reshape(-1) for n in ["A_m", "A_r", "A_inc", "A_amp"]}
inf_pos["A_u0"] = post["u_A"].values[..., 0].reshape(-1)
inf_pos["A_u1"] = post["u_A"].values[..., 1].reshape(-1)
inf_pos.update({n: post[n].values.reshape(-1) for n in ["B_m", "B_r", "B_porb", "B_inc", "B_t0", "B_ecc", "B_w", "B_amp"]})
# w es angular: se recentra alrededor de su media circular para que no se parta en ±180°
w_rad = np.deg2rad(inf_pos["B_w"])
w_c = np.rad2deg(np.arctan2(np.mean(np.sin(w_rad)), np.mean(np.cos(w_rad))))
inf_pos["B_w"] = (inf_pos["B_w"] - w_c + 180.0) % 360.0 + w_c - 180.0
inf_pos["B_h"] = post["B_hk"].values[..., 0].reshape(-1)
inf_pos["B_k_disk"] = post["B_hk"].values[..., 1].reshape(-1)
inf_pos["B_ecosw"] = inf_pos["B_ecc"] * np.cos(np.deg2rad(inf_pos["B_w"]))
inf_pos["B_esinw"] = inf_pos["B_ecc"] * np.sin(np.deg2rad(inf_pos["B_w"]))
inf_pos["B_u0"] = post["u_B"].values[..., 0].reshape(-1)
inf_pos["B_u1"] = post["u_B"].values[..., 1].reshape(-1)

# semieje mayor [AU] por Kepler
G, M_sun, sec_day, AU = 6.6742e-11, 1.99e30, 86400, 1.496e11
P_s = inf_pos["B_porb"] * sec_day
inf_pos["a_AU"] = (G * (inf_pos["A_m"] + inf_pos["B_m"]) * M_sun * P_s ** 2 / (4 * np.pi ** 2)) ** (1 / 3) / AU

Pos = pd.DataFrame(inf_pos)
Pos.to_csv(OUT_DIR / f"{Label}_Posteriors.csv")
Pos.describe().T

In [ ]:
CORNER_KWARGS = dict(
    smooth=0.7, label_kwargs={"fontsize": 14, "labelpad": 8}, title_kwargs={"fontsize": 11, "pad": 10},
    quantiles=[0.16, 0.50, 0.84], plot_density=True, plot_datapoints=True, fill_contours=True,
    show_titles=True, max_n_ticks=4, title_fmt=".3f", use_math_text=True, color="#4C6A4A",
    hist_kwargs={"density": True, "alpha": 0.75, "linewidth": 1.2},
    data_kwargs={"ms": 2.0, "alpha": 0.20}, contour_kwargs={"linewidths": 1.2},
)


def corner_bonito(cols, labels, size, nombre):
    fig = corner.corner(Pos[cols].values, labels=labels, fig=plt.figure(figsize=(size, size)), **CORNER_KWARGS)
    for ax in fig.get_axes():
        ax.tick_params(axis="both", which="major", labelsize=9, length=4, width=0.8, pad=3)
        ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
        for s in ax.spines.values():
            s.set_linewidth(0.8)
    fig.suptitle(f"TIC {TIC}  S{SECTOR}", fontsize=16, fontweight="bold", y=1.0)
    fig.savefig(FIG_DIR / nombre, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()


corner_bonito(["A_m", "A_r", "A_inc", "A_amp", "A_u0", "A_u1"],
              [r"$M_A\ [M_\odot]$", r"$R_A\ [R_\odot]$", r"$i_A\ [^\circ]$", r"amp$_A$", r"$u_{0,A}$", r"$u_{1,A}$"],
              10, f"Corner_{Label}_A.png")
corner_bonito(["B_ecosw", "B_esinw", "B_ecc", "B_w", "B_inc", "B_r"],
              [r"$e\cos\omega$", r"$e\sin\omega$", r"$e$", r"$\omega\ [^\circ]$", r"$i_B\ [^\circ]$", r"$R_B\ [R_\odot]$"],
              11.5, f"Corner_{Label}_excentricidad.png")
corner_bonito(["B_m", "B_r", "B_inc", "a_AU", "B_ecc", "B_w"],
              [r"$M_B\ [M_\odot]$", r"$R_B\ [R_\odot]$", r"$i_B\ [^\circ]$", r"$a\ [AU]$", r"$e_B$", r"$\omega_B\ [^\circ]$"],
              11.5, f"Corner_{Label}_B.png")

## 7) Curva del modelo (percentiles 16/50/84)

In [ ]:
# función del flujo en términos de las variables del posterior (sin reconstruir el modelo)
nombres_in = ["A_inc", "A_amp", "A_r", "A_m", "u_A", "B_inc", "B_amp",
              "B_k" if RADIO_B_COMO_RAZON else "B_r", "B_m", "B_porb", "B_t0", "u_B", "B_ecc", "B_w"]
f_flux = theano.function([model[n] for n in nombres_in], flux_model, on_unused_input="ignore")

n_tot = post.sizes["chain"] * post.sizes["draw"]
sel = np.random.default_rng(1).choice(n_tot, size=min(N_CURVAS, n_tot), replace=False)
planos = {n: post[n].values.reshape((n_tot,) + post[n].shape[2:]) for n in nombres_in}
curvas = np.array([f_flux(*[planos[n][i] for n in nombres_in]) for i in sel])
q16, q50, q84 = np.percentile(curvas, [16, 50, 84], axis=0)

for q, nombre in [(q16, "q16"), (q50, "q50"), (q84, "q84")]:
    pd.DataFrame({"time": t_real, "flux": q}).to_csv(OUT_DIR / f"{Label}_{nombre}.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(t_real, q16, q84, alpha=0.3, label="16--84 percentile")
ax.plot(t_real, q50, lw=2, label="Median model")
ax.scatter(t_real, flux_real, s=8, alpha=0.5, label="Observed")
ax.set_title(Label, fontsize=16, fontweight="bold")
ax.set_xlabel("Time [d]"); ax.set_ylabel("Normalized flux"); ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / f"{Label}_model.png", dpi=300, bbox_inches="tight")
plt.show()

## 8) Tabla de parámetros

In [ ]:
pc = {c: np.percentile(Pos[c], [16, 50, 84]) for c in Pos.columns}


def format_parameter(c, decimals=3):
    lo, med, hi = pc[c]
    return rf"${med:.{decimals}f}^{{+{hi - med:.{decimals}f}}}_{{-{med - lo:.{decimals}f}}}$"


filas = [
    (r"$M_A$", "A_m", 3, r"$M_\odot$"), (r"$R_A$", "A_r", 3, r"$R_\odot$"),
    (r"$u_{0,A}$", "A_u0", 3, ""), (r"$u_{1,A}$", "A_u1", 3, ""),
    (r"$M_B$", "B_m", 3, r"$M_\odot$"), (r"$R_B$", "B_r", 3, r"$R_\odot$"),
    (r"$i$", "B_inc", 3, r"$\mathrm{deg}$"), (r"$P_{\mathrm{orb}}$", "B_porb", 5, r"$\mathrm{d}$"),
    (r"$a$", "a_AU", 4, r"$\mathrm{AU}$"), (r"$t_0$", "B_t0", 5, r"$\mathrm{d}$"),
    (r"$e$", "B_ecc", 4, ""), (r"$\omega$", "B_w", 3, r"$\mathrm{deg}$"),
    (r"$e\cos\omega$", "B_ecosw", 5, ""), (r"$e\sin\omega$", "B_esinw", 4, ""),
    (r"$u_{0,B}$", "B_u0", 3, ""), (r"$u_{1,B}$", "B_u1", 3, ""),
]
Final_info = pd.DataFrame({"Parameter": [f[0] for f in filas],
                           "Value": [format_parameter(f[1], f[2]) for f in filas],
                           "Unit": [f[3] for f in filas]})
display(Final_info)

tabla_png = OUT_DIR / "Tables" / f"{Label}_parameters.png"
tabla_png.parent.mkdir(exist_ok=True)
try:
    import dataframe_image as dfi
    dfi.export(Final_info, str(tabla_png), table_conversion="matplotlib", dpi=300)
    print(f"Tabla guardada en {tabla_png}")
except Exception as e:
    print(f"No se pudo exportar la tabla como imagen: {e}")

# percentiles como variables sueltas (mismos nombres que el ejemplo)
M_a_16, M_a_50, M_a_84 = pc["A_m"]; R_a_16, R_a_50, R_a_84 = pc["A_r"]
amp_a_16, amp_a_50, amp_a_84 = pc["A_amp"]; u0_a_50, u1_a_50 = pc["A_u0"][1], pc["A_u1"][1]
M_b_16, M_b_50, M_b_84 = pc["B_m"]; R_b_16, R_b_50, R_b_84 = pc["B_r"]
i_b_50, p_b_50, t0_b_50 = pc["B_inc"][1], pc["B_porb"][1], pc["B_t0"][1]
e_b_50, w_b_50, amp_b_50 = pc["B_ecc"][1], pc["B_w"][1], pc["B_amp"][1]
u0_b_50, u1_b_50 = pc["B_u0"][1], pc["B_u1"][1]

## 9) Figura final: detrending, curva aplanada y fase

In [ ]:
def phase_fold(t, y, period, t0):
    ph = np.remainder(t - t0, period) / period
    return np.concatenate((ph, ph + 1)), np.concatenate((y, y))


ph_d, f_d = phase_fold(t_real, flux_real, p_b_50, t0_b_50)
ph_m, m50 = phase_fold(t_real, q50, p_b_50, t0_b_50)
_, m16 = phase_fold(t_real, q16, p_b_50, t0_b_50)
_, m84 = phase_fold(t_real, q84, p_b_50, t0_b_50)
fase_d = ph_d - np.floor(ph_d + 0.5)
fase_m = ph_m - np.floor(ph_m + 0.5)
o = np.argsort(fase_m)

plt.rcParams.update({"font.size": 14})
fig = plt.figure(figsize=(8, 12), constrained_layout=True)
gs = GridSpec(3, 1, figure=fig)

ax1 = fig.add_subplot(gs[0])
ax1.scatter(time, flux, s=10, c="black", label="Original")
ax1.plot(time, trend_flux, c="r", lw=2, ls="--", label="Trend")
ax1.set_ylabel("Flux"); ax1.legend(loc="upper left")

ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.scatter(time, flat_flux, c="mediumblue", s=5, label="Flattened")
ax2.set_ylabel("Normalized Flux"); ax2.set_xlabel("Time [days]"); ax2.legend(loc="upper left")

ax3 = fig.add_subplot(gs[2])
ax3.scatter(fase_d, f_d, s=20, color="C1", alpha=0.8, edgecolors="black", lw=0.4, label="Data")
ax3.scatter(fase_m[o], m50[o], color="darkblue", s=5, label="Best Fit")
ax3.fill_between(fase_m[o], m16[o], m84[o], alpha=0.3, color="darkblue", label="Posterior")
ax3.set_xlabel("Phase"); ax3.set_ylabel("Normalized Flux"); ax3.set_xlim(-0.5, 0.5)
ax3.legend(loc="lower right")
ax3.text(-0.48, np.min(m50), f"$P$ = {p_b_50:.4f} days", fontsize=14)

plt.savefig(FIG_DIR / f"Full_{Label}.png", dpi=300)
plt.show()
plt.rcParams.update({"font.size": 10})

## 10) Mapas y animación del sistema (mediana del posterior)

In [ ]:
starry.config.lazy = False
starry.config.quiet = True

A_med = starry.Primary(starry.Map(ydeg=3, udeg=2, amp=amp_a_50), r=R_a_50, m=M_a_50)
A_med.map[1:] = [u0_a_50, u1_a_50]
b_med = starry.Secondary(starry.Map(ydeg=3, udeg=2, inc=i_b_50, amp=amp_b_50), r=R_b_50, m=M_b_50,
                         porb=p_b_50, prot=1.0, t0=t0_b_50, inc=i_b_50, ecc=e_b_50, w=w_b_50)
b_med.map[1:] = [u0_b_50, u1_b_50]
sys_med = starry.System(A_med, b_med)
A_med.map.show()
b_med.map.show()

In [ ]:
if HACER_ANIMACION:
    BG, GREEN, GREEN2, ORANGE = "#F7F4EE", "#273A2F", "#4C6A4A", "#B07D4E"
    npts = 200
    t_an = np.linspace(t0_b_50 - p_b_50 / 2, t0_b_50 + p_b_50 / 2, npts)
    x, y, z = sys_med.position(t_an)
    x, y, z = x - x[0], y - y[0], z - z[0]
    theta_sec = [360.0 / s.prot * (t_an - s.t0) - s.theta0 for s in sys_med.secondaries]
    img = np.array([np.tile(sys_med.primary.map.render(res=300), (npts, 1, 1))]
                   + [s.map.render(theta=theta_sec[i], res=300) for i, s in enumerate(sys_med.secondaries)])
    flux_an = sys_med.flux(t_an)
    t_fold = (t_an - t0_b_50) / p_b_50

    fig = plt.figure(figsize=(7.0, 7.8), facecolor=BG)
    ax = fig.add_axes([0.08, 0.31, 0.84, 0.52])
    ax_xz = fig.add_axes([0.24, 0.83, 0.20, 0.15])
    ax_zy = fig.add_axes([0.56, 0.83, 0.20, 0.15])
    ax_lc = fig.add_axes([0.12, 0.07, 0.76, 0.17])
    for axis, arrs in zip([ax, ax_xz, ax_zy], [(x, y), (x, z), (z, y)]):
        axis.set_facecolor(BG); axis.axis("off")
        R = 1.2 * max(-np.min(arrs), np.max(arrs))
        axis.set_xlim(-R, R); axis.set_ylim(-R, R); axis.set_aspect("equal")
    ax_xz.text(0.02, 0.95, "Side", transform=ax_xz.transAxes, fontsize=11, fontweight="bold", color=GREEN, va="top")
    ax_zy.text(0.02, 0.95, "Orbit", transform=ax_zy.transAxes, fontsize=11, fontweight="bold", color=GREEN, va="top")

    ax_lc.set_facecolor(BG)
    ax_lc.plot(t_fold, flux_an, color=GREEN, lw=1.4)
    (lc,) = ax_lc.plot([t_fold[0]], [flux_an[0]], "o", ms=6, mfc=ORANGE, mec=GREEN)
    phase_line = ax_lc.axvline(t_fold[0], color=ORANGE, lw=1.2, ls="--", alpha=0.75)
    ax_lc.set_xlim(t_fold.min(), t_fold.max())
    ax_lc.set_xlabel("Folded Phase", color=GREEN); ax_lc.set_ylabel("Normalised Flux", color=GREEN)
    ax_lc.grid(True, alpha=0.18)

    n_c = 1 + len(sys_med.secondaries)
    radii = np.array([sys_med.primary.r] + [s.r for s in sys_med.secondaries])
    colores = [ORANGE] + [GREEN2] * len(sys_med.secondaries)
    xz, xy, zy, circ = [None] * n_c, [None] * n_c, [None] * n_c, [None] * n_c
    for i in range(n_c):
        ax_xz.plot(x[i], z[i], color=GREEN2, lw=1, alpha=0.35)
        ax_zy.plot(z[i], y[i], color=GREEN2, lw=1, alpha=0.35)
        xz[i] = ax_xz.scatter(x[i][0], z[i][0], s=35 if i == 0 else 22, color=colores[i], edgecolor=GREEN, zorder=10)
        zy[i] = ax_zy.scatter(z[i][0], y[i][0], s=35 if i == 0 else 22, color=colores[i], edgecolor=GREEN, zorder=10)
        ext = np.array([x[i][0], x[i][0], y[i][0], y[i][0]]) + np.array([-1, 1, -1, 1]) * radii[i]
        xy[i] = ax.imshow(img[i, 0], origin="lower", cmap="magma", extent=ext, interpolation="bilinear",
                          clip_on=False, zorder=z[i][0])
        circ[i] = plt.Circle((x[i][0], y[i][0]), radii[i], fill=False, color=colores[i], lw=1.8, zorder=z[i][0] + 1e-3)
        ax.add_artist(circ[i])
    ax.legend(handles=[Line2D([0], [0], marker="o", color="none", mfc=ORANGE, mec=GREEN, ms=7, label="Primary"),
                       Line2D([0], [0], marker="o", color="none", mfc=GREEN2, mec=GREEN, ms=6, label="Secondary")],
              loc="upper right", frameon=False, fontsize=9, labelcolor=GREEN)
    fig.text(0.5, 0.965, f"Eclipsing Binary Model -- TIC {TIC}", ha="center", fontsize=15, fontweight="bold", color=GREEN)

    def updatefig(k):
        for i in range(n_c):
            xz[i].set_offsets((x[i][k], z[i][k])); zy[i].set_offsets((z[i][k], y[i][k]))
            xy[i].set_extent(np.array([x[i][k], x[i][k], y[i][k], y[i][k]]) + np.array([-1, 1, -1, 1]) * radii[i])
            xy[i].set_zorder(z[i][k]); xy[i].set_data(img[i, k])
            circ[i].center = (x[i][k], y[i][k]); circ[i].set_zorder(z[i][k] + 1e-3)
        lc.set_xdata([t_fold[k]]); lc.set_ydata([flux_an[k]])
        phase_line.set_xdata([t_fold[k], t_fold[k]])
        return xz + xy + zy + circ + [lc, phase_line]

    ani = animation.FuncAnimation(fig, updatefig, interval=30, frames=npts, blit=False)
    plt.close()
    display(HTML(ani.to_html5_video()))
    ani.save(OUT_DIR / f"{Label}_system.mp4", writer=animation.FFMpegWriter(fps=20), dpi=200)

## 11) Rotación (fuera de eclipses)

Lomb-Scargle sobre `flux/trend` quitando las fases de eclipse, con P_orb y t0 del
posterior. Prior de la curva de luz: P_rot = 0.73806 d.

In [ ]:
phase_orb = ((time - t0_b_50) / p_b_50) % 1 - 0.5
mask = (np.abs(phase_orb) > 0.15) & (np.abs(np.abs(phase_orb) - 0.5) > 0.1)
time_clean, flux_clean = time[mask], (flux / trend_flux)[mask]

flux_ls = flux_clean / np.median(flux_clean) - 1
frequency, power = LombScargle(time_clean, flux_ls).autopower(minimum_frequency=0.1, maximum_frequency=5.0)
period = 1 / frequency
best_period = period[np.argmax(power)]
print(f"Periodo dominante: {best_period:.4f} d  (2x = {2 * best_period:.4f} d)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(period, power, "k-")
ax[0].axvline(best_period, color="r", ls="--", label=f"{best_period:.3f} d")
ax[0].axvline(2 * best_period, color="b", ls="--", label=f"{2 * best_period:.3f} d")
ax[0].set(xlabel="Period [days]", ylabel="Power", xlim=(0, 5), title="Lomb-Scargle (sin eclipses)")
ax[0].legend()
fase = ((time_clean - t0_b_50) / best_period) % 1
ax[1].scatter(fase, flux_clean, s=8, color="red", alpha=0.4)
ax[1].set(xlabel="Phase", ylabel="Flux corrected", title=f"Fase con P = {best_period:.4f} d")
plt.tight_layout(); plt.show()